# KRAS inhibitor (MRTX1133) mouse lung Xenium: treatment effect from s, outcome predictor from z

3 control and 4 treated sections, 920 k cells. The per-nodule treatment effect (TE) is the salient composition
shift D([z,s]) − D([z,0]) averaged over a nodule's tumour cells; its main axis (94 %) is conversion toward
Itgb6+ persister tumour epithelium, and nodules with more conversion respond less (independent MAPK MPAS responder
rate, never seen by Haruka).

To find the background that goes with a bad TE, the TE is made s-specific (real − mean of 3 shuffled-s nulls).
Background axis **z-PC2** (alveolar-parenchymal niche: Nqo1/Ptgs2/Ager/Sdc1; endothelium, AT1/AT2, monocytes and
alveolar macrophages vs airway-adjacent ciliated/SMC) predicts the bad TE on held-out sections (3/4; +0.53 vs
responders −0.39) and exists before treatment (control SD ratio 0.94). Interactions: squidpy neighbourhood
enrichment and LIANA+ (lipid exchange Lpl–Sdc1, Apob–Lrp1 at the bad end; complement C3 at the good end).

In [ ]:
import sys, pathlib; sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), pathlib.Path.cwd() / 'notebooks'] if (p / 'repro.py').exists())))
from repro import run, require, show, fetch, fetch_fitted, STAGES, H, WK, PY, PY_VIO, GPU
# Default: download inputs + the paper's fitted Haruka outputs from Hugging Face, then analyses -> figure.
# The Haruka fit step reuses the unpacked fit; HARUKA_REFIT=1 (env) refits from scratch. Auxiliary fits (baselines, seeds) still run.
# Redraw only from saved outputs: STAGES.update(download=False, fit=False, analysis=False)  (or env HARUKA_FIGURE_ONLY=1)
# Rebuild the inputs from the raw public releases instead of Hugging Face: STAGES.update(data=True)
print(H, STAGES)

## 0 Preprocessed data (Hugging Face)

The ready-to-fit inputs are read directly from [C1vy/haruka_v2_data](https://huggingface.co/datasets/C1vy/haruka_v2_data) (cached in `hf_data/`) and linked to the paths the scripts read. The data stage below rebuilds the same files from the raw releases and is off by default.

In [ ]:
fetch('07_kras/mouselung_kras_xenium_clean_noC3.h5ad', H / 'data/mouselung_kras_xenium_clean_noC3.h5ad')

The paper's default Haruka fit, published as fitted AnnData (input + z, s, predicted neighbourhoods; see the dataset card). It is unpacked into the fit folder, so the fit step below reuses it. Set `HARUKA_REFIT=1` before launching to refit instead.

In [ ]:
fetch_fitted('fitted/07_kras/mouselung_kras_xenium_clean_noC3_haruka_fitted.h5ad', H / 'results_kras_te_null/mouselung_kras_xenium_clean_noC3/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0')

## 1 Data: the KRAS Xenium h5ad is distributed as-is (X = normalize+log1p, raw counts in `layers["counts"]`), fetched in section 0

In [ ]:
require(H / 'data/mouselung_kras_xenium_clean_noC3.h5ad')

## 2 Fit (default model; batch = slice, 2000 HVG, saves the model + 3 shuffled-s nulls; ~2.8 h on one GPU)

In [ ]:
run(f'mkdir -p results_kras_te_null && bash bench_kras_xenium/run_kras_te_null_fit.sh {GPU}', 'fit', log='results_kras_te_null/fit.log')
require(H / 'results_kras_te_null/mouselung_kras_xenium_clean_noC3/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0/salient_composition_shift.npy', H / 'results_kras_te_null/mouselung_kras_xenium_clean_noC3/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0/null_shuffled_s_composition_shift_2.npy')

## 3 Independent biological responder (MAPK MPAS, lower quartile of control per tumour subtype)

In [ ]:
run(f'{PY} bench_kras_xenium/define_biological_responder.py', 'analysis', log='bench_kras_xenium/bio_responder.log')

## 4 Nodule treatment effect (panels i–l)

In [ ]:
M = H / 'results_kras_te_null/mouselung_kras_xenium_clean_noC3/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0'; N = ' '.join(str(M / f'null_shuffled_s_composition_shift_{k}.npy') for k in range(3))
NT = H / 'bench_kras_xenium/nodule_te'
run(f'{PY} nodule_te.py {M} refit {N}', 'analysis', cwd=NT, log='run_refit.out')
run(f'{PY} nodule_te.py {M} refit_ownTumEpi {N}', 'analysis', cwd=NT, env={'OWN_TYPE': 'Tumor_epithelial'}, log='run_refit_own.out')

## 5 Which background goes with a bad TE; naming z-PC2; held-out prediction; interactions (panels m–v)

In [ ]:
run(f'{PY} bad_te_background.py', 'analysis', cwd=NT)
run(f'{PY} name_bad_te_z.py', 'analysis', cwd=NT)
run(f'timeout 5400 {PY} zpc2_interactions.py', 'analysis', cwd=NT)

In [ ]:
import pandas as pd
display(pd.read_csv(NT / 'name_bad_te_heldout.csv').round(2))
display(pd.read_csv(NT / 'zpc2_interactions/A_summary.csv').round(3))

## 6 Figure data and the figure

In [ ]:
K = H / 'figures_paper/kras'
run(f'{PY} kras_fig_data.py', 'analysis', cwd=K)
run(f'{PY} make_kras_figure.py', 'figure', cwd=K)

## Figure

In [ ]:
show(H / 'figures_paper/kras/KRAS_inhibitor_full.png')